# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [ ]:
# imports

import os

import openai
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI
import requests


In [ ]:
# constants
MODEL = 'gpt-6-luna'
MODEL_LLAMA = 'llama3.2'

In [ ]:
requests.get("http://localhost:11434").content


In [ ]:
# set up environment
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

ollama_response = requests.get("http://localhost:11434").content

if (api_key and api_key.startswith('sk-proj-') and len(api_key)>10) or ollama_response == b"Ollama is running":
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")

In [ ]:
# here is the question; type over this to ask something new
system_prompt = """
You are a pragmatic, security-first Senior System Design Architect.
We deploy strictly on our own Linux VPS / bare-metal servers (using Docker, Nginx/Caddy, Redis, Postgres, Celery/BullMQ, etc.) — do NOT default to hyperscale cloud services like AWS Lambda, ECS, or GCP unless comparing them as an alternative.

When I ask how to design or scale a system, give me a practical, right-sized architecture for that exact scale with BOTH solutions below:

1. **Capacity & Hardware Sizing (VPS Specs)**:
   - What X concurrent requests actually means in RPS, CPU cores, RAM, GPU/VRAM, and network bandwidth.
   - Exact VPS specs needed (e.g., 4 vCPU, 16GB RAM, or GPU VPS requirements).

2. **Solution A: 100% Open-Source / Self-Hosted on VPS (Zero External Data Sharing)**:
   - Full architecture running inside our VPS (e.g., FastAPI/Node + Redis Queue + Worker pool + Open-Source models like PaddleOCR/Tesseract/vLLM/Ollama).
   - How concurrency is handled without crashing the VPS (async workers, job queues, rate limiting, connection pooling).

3. **Solution B: Hybrid VPS + Frontier API (or Cloud Comparison)**:
   - Running our backend/queue on our VPS while calling Frontier APIs (OpenAI, Anthropic, etc.) with async concurrency (`asyncio`/semaphores, retries, rate-limit handling).
   - Brief comparison against Managed Cloud only for reference.

4. **Security & Zero-PII Leakage**:
   - How to ensure PII (Personally Identifiable Information) never leaks to external Frontier APIs (e.g., local PII scrubbing/redaction on the VPS before external API calls, or routing PII documents strictly to Solution A's local open-source model).
   - VPS hardening (UFW firewall, reverse proxy TLS, secret management, encryption at rest).

5. **Breaking Point & Scaling Up**:
   - At what concurrency this single VPS will bottleneck, and how to scale next (vertical VPS upgrade vs. adding a second worker VPS behind Nginx load balancing).
"""

question = """
How do I manage concurrent 25 request at once for both frontier model and open source module for ocr solution?
"""

In [ ]:
def payload_builder(client, model):
    stream = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
        ],
        stream=True
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [ ]:
# OLLama setup
OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')


In [ ]:
# Get Llama 3.2 to answer
payload_builder(ollama,MODEL_LLAMA)

In [ ]:
#OpenAI to answer
payload_builder(openai, MODEL)

